# Time Series Forecasting — Exercises

Companion to the note [Time Series Forecasting](Time%20Series%20Forecasting.md).

Practise the forecasting workflow: components and stationarity, ACF/PACF, baselines (naive, seasonal naive, drift), exponential smoothing (SES, Holt), AR models fitted by least squares, a Dickey–Fuller unit-root regression, ML with lag features under rolling-origin evaluation, point metrics (MAE, RMSE, MAPE, MASE) and probabilistic forecasts with the pinball loss. Everything is NumPy/scikit-learn (no statsmodels needed).

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.linear_model import LinearRegression, Ridge
rng = np.random.default_rng(0)

def simulate_ar(phi, n, sigma=1.0, c=0.0, seed=0, burn=200):
    """Simulate y_t = c + sum_i phi_i y_{t-i} + eps_t."""
    r = np.random.default_rng(seed)
    phi = np.atleast_1d(phi); p = len(phi)
    y = np.zeros(n + burn)
    eps = sigma * r.normal(size=n + burn)
    for t in range(p, n + burn):
        y[t] = c + phi @ y[t - p:t][::-1] + eps[t]
    return y[burn:]

# Monthly series, 12 years: trend + yearly seasonality + AR(1) noise
t_m = np.arange(144)
y_m = (50 + 0.3 * t_m + 8 * np.sin(2 * np.pi * t_m / 12) + 3 * np.cos(4 * np.pi * t_m / 12)
       + simulate_ar(0.5, 144, sigma=2.0, seed=1))
y_train, y_test = y_m[:120], y_m[120:]   # forecast the last 2 years

## Part A · Concepts

### Exercise 1 · Components and stationarity
*🧠 Concept · ★☆☆*

Which of these series are (weakly) **stationary**? For the non-stationary ones, name the transformation that would help.
1. Daily temperature in Copenhagen over 10 years.
2. White noise $\varepsilon_t\sim\mathcal N(0,1)$.
3. A stock price (approximately a random walk).
4. Monthly airline passengers: upward trend, yearly seasonality whose amplitude grows with the level.
5. An AR(1) process $y_t = 0.7y_{t-1}+\varepsilon_t$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Weak stationarity: constant mean, constant variance, and autocovariance depending only on the lag.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Not stationary (seasonal mean) → seasonal differencing $y_t - y_{t-365}$ or remove a seasonal component (STL).
2. Stationary.
3. Not stationary (unit root, variance grows with $t$) → first differences (returns) $y_t - y_{t-1}$, or log-returns.
4. Not stationary (trend + multiplicative seasonality) → $\log$ to stabilise the variance, then seasonal and/or first differencing (the classic SARIMA "airline model").
5. Stationary since $|\phi|<1$.

</details>

### Exercise 2 · Never shuffle
*🧠 Concept · ★☆☆*

A colleague evaluates a lag-feature gradient-boosting model with ordinary 5-fold shuffled cross-validation and reports an MAE half that of seasonal naive.
On the real future the model is worse than seasonal naive. Explain at least two ways shuffling leaks information, and describe **rolling-origin**
evaluation. See [[Cross-Validation and Model Selection]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

With shuffling, the training folds contain observations from *after* the validation points, and neighbouring points share almost the same lag features.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Look-ahead**: the model trains on $y_{t+1}, y_{t+2},\dots$ and is validated on $y_t$, which is interpolation, not forecasting. Trend and level shifts are "known".
2. **Autocorrelation / overlapping features**: the lag vector of $y_t$ contains $y_{t-1}$, which may be a training *target*, and neighbouring rows are near-duplicates.
3. Rolling statistics or scalers fitted on the whole series also leak future information.

**Rolling-origin** (time-series CV): for origins $T_1 < T_2 < \dots$, train on $y_{1..T_i}$ (expanding or sliding window), forecast $y_{T_i+1..T_i+h}$, average the errors.
Every forecast uses only data available at its origin, exactly as in production (Exercise 17).

</details>

### Exercise 3 · Reading ACF and PACF
*🧠 Concept · ★★☆*

Match the patterns to a model: (a) ACF decays geometrically, PACF cuts off after lag 2; (b) ACF cuts off after lag 1, PACF decays;
(c) ACF decays extremely slowly and stays near 1 for many lags; (d) ACF has spikes at lags 12, 24, 36 in a monthly series.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

AR($p$): the PACF is zero beyond lag $p$. MA($q$): the ACF is zero beyond lag $q$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **AR(2)**: the partial autocorrelation removes indirect effects, so only lags 1 and 2 remain (Exercise 12).
(b) **MA(1)**: $y_t$ and $y_{t-k}$ share no shock for $k>1$.
(c) **Non-stationary** (trend or unit root): difference first (Exercise 16 tests it formally).
(d) **Yearly seasonality**: consider seasonal differencing or a seasonal AR/MA term (SARIMA with $s = 12$).

</details>

### Exercise 4 · Choosing an error metric
*🧠 Concept · ★★☆*

(a) Why is MAPE problematic for intermittent demand (many zeros) and asymmetric? (b) What does sMAPE try to fix? (c) What does MASE $< 1$ mean, and why is it
comparable across series of different scale? (d) When would you prefer RMSE over MAE? (e) What do the pinball loss and CRPS evaluate that MAE cannot?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

MAPE $= \frac{100}{h}\sum|e_t/y_t|$. MASE scales by the in-sample MAE of the naive forecast.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Division by $y_t$: undefined for $y_t = 0$, explodes near 0. It penalises over-forecasts more than under-forecasts (an under-forecast's error is capped at 100%, an over-forecast's is not), so optimising it biases forecasts low.
(b) sMAPE divides by $(|y_t| + |\hat y_t|)/2$ to bound and symmetrise it, but it is still unstable near 0 and not truly symmetric.
(c) MASE $<1$: better than the in-sample one-step naive (or seasonal naive) forecast. Being a ratio of two MAEs on the same series, it is scale-free and can be averaged across series.
(d) RMSE when large errors are disproportionately costly; it is minimised by the conditional **mean**, MAE by the **median**.
(e) Pinball (quantile) loss evaluates a predicted quantile, CRPS a whole predictive distribution: they reward calibrated and sharp *uncertainty*, not just a point forecast.

</details>

## Part B · By hand

### Exercise 5 · Naive vs seasonal naive
*✍️ By hand · ★☆☆*

Quarterly sales for two years: $y = (10, 20, 30, 15,\ 12, 22, 33, 16)$. The next four quarters turn out to be $(13, 24, 35, 18)$.
Compute the 4-step forecasts of the **naive** method (last value) and the **seasonal naive** method (value one season ago, $m=4$).
Then compute the MAE and RMSE of naive, and the MAE and MAPE (in %) of seasonal naive.

In [ ]:
mae_naive = None
rmse_naive = None
mae_snaive = None
mape_snaive = None  # in percent

_y = np.array([10, 20, 30, 15, 12, 22, 33, 16.]); _f = np.array([13, 24, 35, 18.])
check('MAE naive', mae_naive, np.mean(np.abs(_f - _y[-1])))
check('RMSE naive', rmse_naive, np.sqrt(np.mean((_f - _y[-1]) ** 2)), tol=1e-3)
check('MAE seasonal naive', mae_snaive, np.mean(np.abs(_f - _y[-4:])))
check('MAPE seasonal naive', mape_snaive, 100 * np.mean(np.abs(_f - _y[-4:]) / _f), tol=1e-2)

<details>
<summary><b>💡 Hint</b></summary>

Naive: $(16,16,16,16)$. Seasonal naive: the last season $(12,22,33,16)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Naive errors: $(3, 8, 19, 2)$ → MAE $= 32/4 = 8$, RMSE $= \sqrt{(9+64+361+4)/4} = \sqrt{109.5}\approx 10.46$.
Seasonal naive errors: $(1, 2, 2, 2)$ → MAE $= 1.75$, MAPE $= \frac{100}{4}\big(\frac1{13}+\frac2{24}+\frac2{35}+\frac2{18}\big)\approx 8.21\%$.
On strongly seasonal data the seasonal naive baseline is hard to beat, which is why the note says *always compare*.

```python
mae_naive = 8.0
rmse_naive = np.sqrt(109.5)
mae_snaive = 1.75
mape_snaive = 100 / 4 * (1/13 + 2/24 + 2/35 + 2/18)
```

</details>

### Exercise 6 · Simple exponential smoothing by hand
*✍️ By hand · ★☆☆*

SES: $\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$, forecast $\hat y_{T+h\mid T} = \ell_T$ for all $h$. With $\alpha = 0.5$, $\ell_0 = 10$ and
$y = (10, 12, 11, 15)$, compute $\ell_1,\dots,\ell_4$ and the forecast. Which weight does SES put on $y_4$, $y_3$, $y_2$?

In [ ]:
levels = None     # [l1, l2, l3, l4]
ses_forecast = None

_l, _L = 10.0, []
for _v in [10, 12, 11, 15.]:
    _l = 0.5 * _v + 0.5 * _l; _L.append(_l)
check('levels', levels, _L)
check('forecast', ses_forecast, _L[-1])

<details>
<summary><b>💡 Hint</b></summary>

Unroll: $\ell_T = \alpha y_T + \alpha(1-\alpha)y_{T-1} + \alpha(1-\alpha)^2y_{T-2} + \dots$

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ell_1 = 10$, $\ell_2 = 0.5\cdot12 + 0.5\cdot10 = 11$, $\ell_3 = 0.5\cdot 11 + 0.5\cdot 11 = 11$, $\ell_4 = 0.5\cdot15 + 0.5\cdot11 = 13$. Forecast: 13 for every horizon.
Weights $\alpha(1-\alpha)^j$: $0.5$ on $y_4$, $0.25$ on $y_3$, $0.125$ on $y_2$: exponentially decaying, hence the name. $\alpha\to1$ gives the naive method, $\alpha\to0$ the overall mean.

```python
levels = [10, 11, 11, 13]
ses_forecast = 13
```

</details>

### Exercise 7 · AR(1) properties and forecasts
*✍️ By hand · ★★☆*

$y_t = c + \phi y_{t-1} + \varepsilon_t$ with $c = 1$, $\phi = 0.8$, $\varepsilon_t\sim\mathcal N(0, 1)$ i.i.d.
(a) Compute the stationary mean $\mu$, variance $\gamma_0$ and the autocorrelation at lag 2. (b) If $y_T = 7$, compute the forecasts $\hat y_{T+1}$ and $\hat y_{T+2}$.
What does $\hat y_{T+h}$ converge to as $h\to\infty$?

In [ ]:
ar_mean = None
ar_var = None
ar_rho2 = None
fc1 = None
fc2 = None

_y = simulate_ar(0.8, 400_000, c=1.0, seed=5)
check('mean (vs long simulation)', ar_mean, _y.mean(), tol=0.05)
check('variance (vs long simulation)', ar_var, _y.var(), tol=0.05)
check('rho(2) (vs long simulation)', ar_rho2, np.corrcoef(_y[2:], _y[:-2])[0, 1], tol=0.01)
check('1-step forecast', fc1, 1 + 0.8 * 7)
check('2-step forecast', fc2, 1 + 0.8 * (1 + 0.8 * 7))

<details>
<summary><b>💡 Hint 1</b></summary>

Stationarity: $\mathbb E[y_t] = \mathbb E[y_{t-1}] = \mu$ and $\operatorname{Var}(y_t) = \operatorname{Var}(y_{t-1})$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\rho(k) = \phi^k$. Forecasts: replace future shocks by their mean 0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\mu = c/(1-\phi) = 5$; $\gamma_0 = \sigma^2/(1-\phi^2) = 1/0.36\approx 2.778$; $\rho(2) = \phi^2 = 0.64$.
(b) $\hat y_{T+1} = 1 + 0.8\cdot7 = 6.6$; $\hat y_{T+2} = 1 + 0.8\cdot6.6 = 6.28$. In general $\hat y_{T+h} = \mu + \phi^h(y_T - \mu)\to\mu = 5$: forecasts revert to the mean,
and the forecast variance $\sigma^2\sum_{j<h}\phi^{2j}$ grows to $\gamma_0$.

```python
ar_mean = 5.0
ar_var = 1 / 0.36
ar_rho2 = 0.64
fc1 = 6.6
fc2 = 6.28
```

</details>

### Exercise 8 · What differencing does
*✍️ By hand · ★★☆*

(a) $y_t = 3 + 2t + s_t$ where $s_t$ is a fixed pattern with period 4 ($s_t = s_{t-4}$). Compute the seasonal difference $\nabla_4y_t = y_t - y_{t-4}$.
(b) For the **random walk** $y_t = y_{t-1} + \varepsilon_t$, $y_0 = 0$, $\varepsilon_t\sim\mathcal N(0, 0.5^2)$: what is $\operatorname{Var}(y_{100})$, and what is the first difference?
(c) Why is a random walk not stationary although its first difference is white noise?

In [ ]:
seasonal_diff = None
var_rw_100 = None

_t = np.arange(40); _s = np.tile([5., -2, 1, -4], 10); _y = 3 + 2 * _t + _s
check('seasonal difference', seasonal_diff, np.unique(np.round(_y[4:] - _y[:-4], 10))[0])
_rw = np.cumsum(0.5 * np.random.default_rng(0).normal(size=(20000, 100)), axis=1)
check('Var(y_100) (vs simulation)', var_rw_100, _rw[:, -1].var(), tol=1.0)

<details>
<summary><b>💡 Hint</b></summary>

The periodic part cancels in $y_t - y_{t-4}$; the trend leaves $2\cdot 4$. For the random walk, $y_{100} = \sum_{t=1}^{100}\varepsilon_t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\nabla_4y_t = 2t - 2(t-4) + s_t - s_{t-4} = 8$: seasonal differencing removes the seasonal pattern *and* turns a linear trend into a constant.
(b) $\operatorname{Var}(y_{100}) = 100\cdot0.25 = 25$; $\Delta y_t = \varepsilon_t$, white noise.
(c) The variance $t\sigma^2$ grows with time (and $\operatorname{Cov}(y_t,y_{t+k}) = t\sigma^2$ depends on $t$), violating stationarity. The process is integrated of order 1, I(1): the "I" in ARIMA.

```python
seasonal_diff = 8
var_rw_100 = 25.0
```

</details>

### Exercise 9 · MASE by hand
*✍️ By hand · ★★☆*

Training data $y = (10, 12, 11, 15, 14)$; a model forecasts $(16, 16)$ for the next two periods, which turn out to be $(13, 18)$.
Compute the scale $Q = \frac{1}{T-1}\sum_{t=2}^T|y_t - y_{t-1}|$ (in-sample one-step naive MAE) and $\text{MASE} = \text{MAE}/Q$. Is the model better than naive?

In [ ]:
Q_scale = None
mase_hand = None

_tr = np.array([10, 12, 11, 15, 14.]); _q = np.mean(np.abs(np.diff(_tr)))
check('scale Q', Q_scale, _q)
check('MASE', mase_hand, np.mean(np.abs(np.array([13, 18.]) - 16)) / _q)

<details>
<summary><b>💡 Hint</b></summary>

In-sample differences: $|2|, |-1|, |4|, |-1|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$Q = (2 + 1 + 4 + 1)/4 = 2$; MAE $= (3 + 2)/2 = 2.5$; MASE $= 2.5/2 = 1.25 > 1$: worse than what the naive method achieves (in-sample) one step ahead.
(Comparing a 2-step forecast with a 1-step benchmark is a bit harsh, but this is the standard definition of Hyndman & Koehler.)

```python
Q_scale = 2.0
mase_hand = 1.25
```

</details>

### Exercise 10 · The pinball loss
*✍️ By hand · ★★☆*

The pinball loss of a quantile forecast $q$ at level $\tau$ is $\rho_\tau(y, q) = \tau(y - q)$ if $y\ge q$ and $(1-\tau)(q - y)$ otherwise.
(a) With $\tau = 0.9$ and $q = 100$, compute the loss for $y = 110$ and for $y = 90$. (b) Show that the expected pinball loss
$\mathbb E_Y[\rho_\tau(Y, q)]$ is minimised when $q$ is the $\tau$-quantile of $Y$.

In [ ]:
pinball_110 = None
pinball_90 = None

def _pin(y, q, tau): return max(tau * (y - q), (tau - 1) * (y - q))
check('loss at y=110', pinball_110, _pin(110, 100, 0.9))
check('loss at y=90', pinball_90, _pin(90, 100, 0.9))

<details>
<summary><b>💡 Hint</b></summary>

Differentiate $\int\rho_\tau(y,q)\,dF(y)$ w.r.t. $q$: the part with $y<q$ contributes $(1-\tau)F(q)$, the part with $y\ge q$ contributes $-\tau(1-F(q))$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $y = 110 \ge q$: $0.9\cdot10 = 9$. $y = 90 < q$: $0.1\cdot 10 = 1$. Under-predicting a 90% quantile is 9× as costly as over-predicting, so the optimal $q$ sits high.
(b) $\frac{d}{dq}\mathbb E[\rho_\tau] = (1-\tau)F(q) - \tau(1 - F(q)) = F(q) - \tau = 0 \Rightarrow F(q) = \tau$. For $\tau = 0.5$ this is (half) the absolute error, minimised by the median.
Averaging the pinball loss over many $\tau$ approximates the CRPS.

```python
pinball_110 = 9.0
pinball_90 = 1.0
```

</details>

## Part C · Code

### Exercise 11 · Sample ACF
*💻 Code · ★☆☆*

Implement `acf(y, nlags)` returning $r_0,\dots,r_{\text{nlags}}$ with
$r_k = \frac{\sum_{t=1}^{T-k}(y_t - \bar y)(y_{t+k} - \bar y)}{\sum_{t=1}^T(y_t-\bar y)^2}$. Apply it to the monthly series `y_m` and to its seasonal difference.
What do you see at lag 12?

In [ ]:
def acf(y, nlags):
    # TODO
    return None

acf_m = acf(y_m, 36)

_y = simulate_ar(0.6, 2000, seed=3); _yc = _y - _y.mean()
_ref = np.correlate(_yc, _yc, 'full')[len(_y) - 1:][:11] / np.sum(_yc ** 2)
check('acf vs np.correlate', acf(_y, 10), _ref)

<details>
<summary><b>💡 Hint</b></summary>

`yc = y - y.mean()`; `np.array([yc[:len(y)-k] @ yc[k:] for k in range(nlags+1)]) / (yc @ yc)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The ACF of `y_m` decays slowly (trend) with bumps at lags 12, 24, 36 (seasonality). After seasonal differencing $y_t - y_{t-12}$ the trend becomes a constant and the
pattern is mostly gone, leaving the short-memory AR noise structure (plus a negative spike near lag 12, typical of over-differencing the noise).
Approximate 95% bands for white noise are $\pm1.96/\sqrt T$.

```python
def acf(y, nlags):
    yc = np.asarray(y) - np.mean(y)
    return np.array([yc[:len(yc) - k] @ yc[k:] for k in range(nlags + 1)]) / (yc @ yc)

acf_m = acf(y_m, 36)
print('ACF at lags 1, 6, 12, 24:', np.round(acf_m[[1, 6, 12, 24]], 2))
print('after seasonal differencing:', np.round(acf(y_m[12:] - y_m[:-12], 12)[[1, 6, 12]], 2))
```

</details>

### Exercise 12 · PACF by regression
*💻 Code · ★★☆*

The partial autocorrelation at lag $k$ is the last coefficient $\hat\phi_{kk}$ in the regression of $y_t$ on $y_{t-1},\dots,y_{t-k}$ (with intercept).
Implement `pacf(y, nlags)` this way using `np.linalg.lstsq`, and apply it to a simulated AR(2) process with $\phi = (0.5, 0.3)$ ($n = 5000$).
Store the result in `pacf_ar2` (lags $0..6$, with `pacf[0] = 1`). Where does it cut off?

In [ ]:
def lag_matrix(y, p):
    """Rows t = p..T-1: [y_{t-1}, ..., y_{t-p}]; returns (X, target)."""
    y = np.asarray(y)
    X = np.column_stack([y[p - i:len(y) - i] for i in range(1, p + 1)])
    return X, y[p:]

def pacf(y, nlags):
    # TODO
    return None

y_ar2 = simulate_ar([0.5, 0.3], 5000, seed=2)
pacf_ar2 = pacf(y_ar2, 6)

if pacf_ar2 is not None:
    _ref = [1.0] + [LinearRegression().fit(*lag_matrix(y_ar2, k)).coef_[-1] for k in range(1, 7)]
    check('PACF vs sklearn regressions', pacf_ar2, _ref)
    check('cut-off after lag 2', np.all(np.abs(pacf_ar2[3:]) < 2 / np.sqrt(5000)), True)
else:
    check('PACF', None, 1)

<details>
<summary><b>💡 Hint</b></summary>

For each $k$: `X, t = lag_matrix(y, k)`; solve with a column of ones prepended; take the coefficient of $y_{t-k}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\hat\phi_{22}\approx 0.3 = \phi_2$ and the PACF is inside $\pm2/\sqrt n$ beyond lag 2: an AR(2) signature. The lag-1 PACF equals the ACF at lag 1,
$\rho_1 = \phi_1/(1-\phi_2)\approx 0.71$. (statsmodels' default uses Yule–Walker equations instead of OLS; the values are nearly identical for long series.)

```python
def lag_matrix(y, p):
    y = np.asarray(y)
    X = np.column_stack([y[p - i:len(y) - i] for i in range(1, p + 1)])
    return X, y[p:]

def pacf(y, nlags):
    out = [1.0]
    for k in range(1, nlags + 1):
        X, target = lag_matrix(y, k)
        beta = np.linalg.lstsq(np.c_[np.ones(len(X)), X], target, rcond=None)[0]
        out.append(beta[-1])
    return np.array(out)

y_ar2 = simulate_ar([0.5, 0.3], 5000, seed=2)
pacf_ar2 = pacf(y_ar2, 6)
print(np.round(pacf_ar2, 3))
```

</details>

### Exercise 13 · Baselines and metrics on the monthly series
*💻 Code · ★☆☆*

Implement the metrics `mae`, `rmse`, `mape`, `mase(y_true, y_pred, y_train, m)` (scale = in-sample MAE of the **seasonal** naive with period $m$) and the
24-step baselines `naive`, `seasonal_naive(m=12)`, `drift` (line through first and last training points) and `mean`.
Forecast `y_test` from `y_train` and store a dict `scores[name] = (MAE, MASE)` with $m = 12$.

In [ ]:
def mae(y, f):  return None
def rmse(y, f): return None
def mape(y, f): return None
def mase(y, f, y_train, m=1): return None

def baselines(y_train, h, m=12):
    # TODO: return {'naive': ..., 'seasonal_naive': ..., 'drift': ..., 'mean': ...}, each an array of length h
    return None

scores = None

check('MASE on the by-hand example (m=1)', mase(np.array([13, 18.]), np.array([16, 16.]), np.array([10, 12, 11, 15, 14.]), 1), 1.25)
if scores is not None:
    check('seasonal naive beats naive and mean', scores['seasonal_naive'][0] < min(scores['naive'][0], scores['mean'][0]), True)
else:
    check('seasonal naive beats naive and mean', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Seasonal naive for horizon $h$: `y_train[-m:][np.arange(h) % m]`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Drift: `y_train[-1] + np.arange(1, h+1) * (y_train[-1] - y_train[0]) / (len(y_train) - 1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Here: seasonal naive MAE ≈ 5.15, drift ≈ 5.73, naive ≈ 6.49, mean ≈ 21.5. Seasonal naive captures the yearly pattern but not the trend (it lags one year behind the level); drift captures the trend but not the season; naive and mean capture neither.
The MASE values tell you directly how each compares with the seasonal-naive *in-sample* benchmark. Any real model must beat the best of these.

```python
def mae(y, f):  return np.mean(np.abs(np.asarray(y) - f))
def rmse(y, f): return np.sqrt(np.mean((np.asarray(y) - f) ** 2))
def mape(y, f): return 100 * np.mean(np.abs((np.asarray(y) - f) / y))
def mase(y, f, y_train, m=1):
    y_train = np.asarray(y_train)
    return mae(y, f) / np.mean(np.abs(y_train[m:] - y_train[:-m]))

def baselines(y_train, h, m=12):
    T = len(y_train)
    return {
        'naive': np.full(h, y_train[-1]),
        'seasonal_naive': y_train[-m:][np.arange(h) % m],
        'drift': y_train[-1] + np.arange(1, h + 1) * (y_train[-1] - y_train[0]) / (T - 1),
        'mean': np.full(h, y_train.mean()),
    }

fc_base = baselines(y_train, 24)
scores = {k: (mae(y_test, f), mase(y_test, f, y_train, 12)) for k, f in fc_base.items()}
for k, (a, b) in scores.items():
    print(f'{k:15s} MAE={a:6.2f}  MASE={b:5.2f}')
```

</details>

### Exercise 14 · SES and Holt's linear trend
*💻 Code · ★★☆*

Implement `ses(y, alpha, h)` (with $\ell_0 = y_1$) and Holt's method `holt(y, alpha, beta, h)`:
$$\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + b_{t-1}),\quad b_t = \beta(\ell_t - \ell_{t-1}) + (1-\beta)b_{t-1},\quad \hat y_{T+h} = \ell_T + hb_T,$$
initialised with $\ell_0 = y_1$, $b_0 = y_2 - y_1$. Check SES against its explicit weighted-average form and Holt on a noiseless line.

In [ ]:
def ses(y, alpha, h):
    # TODO: return array of h identical forecasts
    return None

def holt(y, alpha, beta, h):
    # TODO
    return None

_y = rng.normal(size=30) + 5; _a = 0.3
_w = _a * (1 - _a) ** np.arange(29, -1, -1); _w[0] = (1 - _a) ** 29      # weight of l0 = y_1 folded into the first weight
check('SES = weighted average', None if ses(_y, _a, 3) is None else ses(_y, _a, 3)[0], _w @ _y)
check('Holt extrapolates a line exactly', holt(2 + 0.5 * np.arange(20), 0.5, 0.3, 5), 2 + 0.5 * np.arange(20, 25))

<details>
<summary><b>💡 Hint</b></summary>

Loop over $t$; for SES the forecast is `np.full(h, level)`. For Holt keep the previous level before updating the trend.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SES is the exponentially weighted average from Exercise 6 (the initial level absorbs the remaining weight $(1-\alpha)^{T-1}$ on $y_1$). Holt adds a smoothed slope, so its forecasts are a line;
on a perfect line both components are exact from the start. Holt–Winters adds a seasonal component $s_t$ with its own smoothing parameter $\gamma$; ETS models are the statistical
(state-space) formulation of these recursions and estimate $\alpha,\beta,\gamma$ by maximum likelihood.

```python
def ses(y, alpha, h):
    level = y[0]
    for v in y[1:]:
        level = alpha * v + (1 - alpha) * level
    return np.full(h, level)

def holt(y, alpha, beta, h):
    level, trend = y[0], y[1] - y[0]
    for v in y[1:]:
        prev = level
        level = alpha * v + (1 - alpha) * (level + trend)
        trend = beta * (level - prev) + (1 - beta) * trend
    return level + trend * np.arange(1, h + 1)
```

</details>

### Exercise 15 · AR(p) by least squares and recursive forecasting
*💻 Code · ★★☆*

Write `fit_ar(y, p)` (OLS with intercept on the lag matrix from Exercise 12, returns `(c, phi)`) and `forecast_ar(y, c, phi, h)` which forecasts **recursively**,
feeding each forecast back as a lag. (a) Fit an AR(2) to `y_ar2` and store `phi_hat`. (b) For the monthly series, fit an AR(13) to the **seasonally differenced** training data
$z_t = y_t - y_{t-12}$, forecast $z$ for 24 steps, undo the differencing, and store the MAE on `y_test` in `mae_sar`.

In [ ]:
def fit_ar(y, p):
    return None, None

def forecast_ar(y, c, phi, h):
    return None

phi_hat = None
mae_sar = None

check('AR(2) coefficients close to the true (0.5, 0.3)', phi_hat, [0.5, 0.3], tol=0.05)
if mae_sar is not None:
    _sn = y_train[-12:][np.arange(24) % 12]
    check('beats seasonal naive', mae_sar < np.mean(np.abs(y_test - _sn)), True)
else:
    check('beats seasonal naive', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Recursive forecasting: `hist = list(y)`; repeat $h$ times: `nxt = c + phi @ hist[-1:-p-1:-1]`; append.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Undo seasonal differencing step by step: $\hat y_{T+k} = \hat z_{T+k} + \hat y_{T+k-12}$, where $\hat y$ falls back to observed values for $T+k-12\le T$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The OLS estimates are close to $(0.5, 0.3)$. On the monthly series, seasonal differencing removes the seasonality and turns the trend into a constant ($12\times0.3 = 3.6$),
which the AR intercept captures, so the ARIMA-style model improves on seasonal naive, which ignores the trend (test MAE ≈ 1.9 vs ≈ 5.15). This is essentially a SARIMA$(13,0,0)\times(0,1,0)_{12}$ fitted by OLS.

```python
def fit_ar(y, p):
    X, target = lag_matrix(y, p)
    beta = np.linalg.lstsq(np.c_[np.ones(len(X)), X], target, rcond=None)[0]
    return beta[0], beta[1:]

def forecast_ar(y, c, phi, h):
    hist = list(y); p = len(phi)
    for _ in range(h):
        hist.append(c + phi @ np.array(hist[-1:-p - 1:-1]))
    return np.array(hist[len(y):])

phi_hat = fit_ar(y_ar2, 2)[1]
z = y_train[12:] - y_train[:-12]
c_z, phi_z = fit_ar(z, 13)
z_fc = forecast_ar(z, c_z, phi_z, 24)
hist = list(y_train)
for k in range(24):
    hist.append(z_fc[k] + hist[-12])
y_fc_sar = np.array(hist[120:])
mae_sar = np.mean(np.abs(y_test - y_fc_sar))
print('phi_hat =', np.round(phi_hat, 3), '  MAE seasonal-AR =', round(mae_sar, 2))
```

</details>

### Exercise 16 · A Dickey–Fuller regression
*💻 Code · ★★★*

The (non-augmented) Dickey–Fuller test regresses $\Delta y_t = \alpha + \gamma y_{t-1} + e_t$ and uses the t-statistic of $\hat\gamma$; $H_0$: unit root ($\gamma = 0$).
Under $H_0$ the statistic does **not** follow a t-distribution; the 5% critical value (with constant) is about $-2.86$.
Implement `df_stat(y)` and compute it for a random walk and for an AR(1) with $\phi = 0.9$ (both $n = 500$, seed 0). Store `df_rw`, `df_ar`.

In [ ]:
def df_stat(y):
    # TODO: OLS of diff(y) on [1, y_{t-1}]; return gamma_hat / se(gamma_hat)
    return None

y_rw = np.cumsum(np.random.default_rng(0).normal(size=500))
y_ar09 = simulate_ar(0.9, 500, seed=0)
df_rw = df_stat(y_rw)
df_ar = df_stat(y_ar09)

check('random walk: cannot reject the unit root', None if df_rw is None else df_rw > -2.86, True)
check('AR(0.9): reject the unit root', None if df_ar is None else df_ar < -2.86, True)

<details>
<summary><b>💡 Hint 1</b></summary>

$\hat\beta = (X^\top X)^{-1}X^\top\Delta y$, residual variance $s^2 = \lVert e\rVert^2/(n-2)$, $\operatorname{se}(\hat\gamma) = \sqrt{s^2[(X^\top X)^{-1}]_{11}}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$\Delta y_t = y_t - y_{t-1}$ has length $n-1$; regress it on $y_{t-1}$ = `y[:-1]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\gamma = \phi - 1$: for a random walk $\gamma = 0$, for the AR(0.9) $\gamma = -0.1$. The random walk's statistic is above $-2.86$ (≈ −1.07, no rejection), the AR(0.9)'s well below it (≈ −5.89).
Under $H_0$ the regressor $y_{t-1}$ is itself non-stationary, so the usual central limit theorem fails and the statistic follows the Dickey–Fuller distribution (skewed left),
hence the special critical values. The **augmented** test adds lagged $\Delta y_{t-j}$ terms to absorb serial correlation; KPSS reverses the hypotheses (H₀: stationary).

```python
def df_stat(y):
    y = np.asarray(y)
    dy = np.diff(y)
    X = np.c_[np.ones(len(dy)), y[:-1]]
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ X.T @ dy
    e = dy - X @ beta
    s2 = e @ e / (len(dy) - 2)
    return beta[1] / np.sqrt(s2 * XtX_inv[1, 1])

y_rw = np.cumsum(np.random.default_rng(0).normal(size=500))
y_ar09 = simulate_ar(0.9, 500, seed=0)
df_rw = df_stat(y_rw)
df_ar = df_stat(y_ar09)
print('DF random walk:', round(df_rw, 2), '  DF AR(0.9):', round(df_ar, 2))
```

</details>

### Exercise 17 · Lag features + rolling-origin evaluation
*💻 Code · ★★★*

Build a **direct** ML forecaster: features at time $t$ are lags $y_{t-1},\dots,y_{t-12}$ plus the time index $t$ and month-of-year one-hot; target $y_t$.
Fit `Ridge(alpha=1.0)` and forecast **one step ahead**. Evaluate with **rolling origin** on the whole series `y_m`: for origins $T = 96, 97, \dots, 143$, train only on targets
with index $< T$ and predict $y_T$. Store the mean absolute error in `mae_ridge_ro` and the seasonal naive one-step MAE over the same origins in `mae_snaive_ro`.

In [ ]:
def make_features(y, t_idx, n_lags=12):
    """Feature row for predicting y[t] (uses only y[:t])."""
    # TODO
    return None

mae_ridge_ro = None
mae_snaive_ro = None

if mae_ridge_ro is not None:
    check('ridge with lags beats seasonal naive (one step)', mae_ridge_ro < mae_snaive_ro, True)
    check('seasonal naive MAE', mae_snaive_ro, np.mean(np.abs(y_m[96:144] - y_m[84:132])))
else:
    check('ridge with lags beats seasonal naive (one step)', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

`np.r_[y[t-12:t][::-1], t, np.eye(12)[t % 12]]` is one feature row. Only build rows for $t\ge 12$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Inside the loop over origins `T`: `X_tr = [make_features(y_m, t) for t in range(12, T)]`, `y_tr = y_m[12:T]`, predict `make_features(y_m, T)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each origin retrains on the past only, so the evaluation mimics production. The ridge model beats seasonal naive one step ahead (MAE ≈ 1.82 vs ≈ 3.85) because the lags carry the AR noise
(this month is high → next month is likely high too) and the time index carries the trend. With a tree model (LightGBM, [[Ensemble Methods]]) note that trees cannot
extrapolate a trend: detrend or difference first. For $h$-step forecasts either train one model per horizon (direct) or feed predictions back (recursive, as in Exercise 15).

```python
def make_features(y, t_idx, n_lags=12):
    return np.r_[y[t_idx - n_lags:t_idx][::-1], t_idx, np.eye(12)[t_idx % 12]]

errs_ridge, errs_sn = [], []
for T in range(96, 144):
    X_tr = np.array([make_features(y_m, t) for t in range(12, T)])
    model = Ridge(alpha=1.0).fit(X_tr, y_m[12:T])
    pred = model.predict(make_features(y_m, T)[None])[0]
    errs_ridge.append(abs(y_m[T] - pred))
    errs_sn.append(abs(y_m[T] - y_m[T - 12]))
mae_ridge_ro, mae_snaive_ro = np.mean(errs_ridge), np.mean(errs_sn)
print('rolling-origin MAE  ridge:', round(mae_ridge_ro, 2), '  seasonal naive:', round(mae_snaive_ro, 2))
```

</details>

### Exercise 18 · Prediction intervals, coverage and pinball loss
*💻 Code · ★★☆*

Turn a point forecaster into a probabilistic one: using a long AR(1) series (`simulate_ar(0.7, 3000, seed=4)`), fit AR(1) on the first 2000 points,
compute the in-sample one-step residuals, and form 80% intervals $[\hat y + q_{0.1}, \hat y + q_{0.9}]$ from the empirical residual quantiles.
On the last 1000 points compute the **empirical coverage** (`coverage`) and the mean pinball loss of the 0.9-quantile forecast (`pinball_09`).
Implement `pinball(y, q, tau)` yourself.

In [ ]:
def pinball(y, q, tau):
    # TODO: mean pinball loss
    return None

y_q = simulate_ar(0.7, 3000, seed=4)
coverage = None
pinball_09 = None

check('pinball on the by-hand example', pinball(np.array([110., 90.]), 100.0, 0.9), 5.0)
check('80% interval covers ~80%', None if coverage is None else abs(coverage - 0.8) < 0.04, True)

<details>
<summary><b>💡 Hint 1</b></summary>

One-step forecasts: $\hat y_t = \hat c + \hat\phi y_{t-1}$, both in-sample (residuals) and on the test part (each uses the true previous value).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.quantile(resid, [0.1, 0.9])` gives the offsets.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Coverage comes out close to 80% (0.82 here) because the residual distribution is stable (stationary Gaussian noise). In real data, residual quantiles from in-sample fits are usually
too narrow (overfitting, regime changes); conformal prediction on a held-out calibration window fixes the coverage. For multi-step horizons the interval must widen
(Exercise 7: forecast variance grows with $h$). The pinball loss rewards intervals that are both calibrated and sharp.

```python
def pinball(y, q, tau):
    d = np.asarray(y) - q
    return np.mean(np.maximum(tau * d, (tau - 1) * d))

y_q = simulate_ar(0.7, 3000, seed=4)
c_q, phi_q = fit_ar(y_q[:2000], 1)
resid = y_q[1:2000] - (c_q + phi_q[0] * y_q[:1999])
lo, hi = np.quantile(resid, [0.1, 0.9])
pred_test = c_q + phi_q[0] * y_q[1999:2999]
y_true = y_q[2000:3000]
coverage = np.mean((y_true >= pred_test + lo) & (y_true <= pred_test + hi))
pinball_09 = pinball(y_true, pred_test + hi, 0.9)
print('coverage:', coverage, '  pinball(0.9):', round(pinball_09, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Time Series Forecasting](Time%20Series%20Forecasting.md).*